## Running WorldCereal classification models locally

This notebook demonstrates how you can apply trained WorldCereal cropland/crop type classification models on patches of preprocessed inputs.

### Why?

Having a set of local patches with preprocessed inputs available comes in handy during the development of a custom crop type model. It allows you to quickly test different model set-ups, as each time you have trained a new model, you can immediately apply it to the same set of patches and check for improvements. By not having to deploy and run the model on CDSE, this drastically reduces the time required to get to your ideal crop model!

### How does it work?

1. Make sure you acquire a set of patches representative for your area/season of interest by following the steps in <a href="https://github.com/WorldCereal/worldcereal-classification/blob/main/notebooks/worldcereal_preprocessed_inputs.ipynb" target="_blank">this notebook</a>.

2. Train your model following the steps outlined in the full training workflow in our <a href="https://github.com/WorldCereal/worldcereal-classification/tree/main/notebooks/worldcereal_classification_app.ipynb" target="_blank">classification app</a>. You need to complete at least steps 1-6 (Train Classifier) here, up to the point where you have your model saved locally as a .zip file.

3. Simply then follow the steps below to apply your model to any of your prepared patches.

### Step 1: Select model to run

In the next cell we check which locally trained models are currently available on your machine.

In [ ]:
import glob
from pathlib import Path

model_dir = Path("./downstream_heads")
available_models = glob.glob(str(model_dir / "*"))
available_models = [Path(m).stem for m in available_models if Path(m).is_dir()]
# print with a line break between each model
print("Available models:")
for model in available_models:
    print(model)

The next cell asks you to provide the name of the model you wish to use.<br>

NOTE for experienced users: in case this particular model has been trained with a seasonal Presto model other than the default WorldCereal model, make sure to specify the seasonal_model_zip_path in the next cell.

In [ ]:
from worldcereal.openeo.parameters import DEFAULT_SEASONAL_MODEL_URL
seasonal_model_zip_path = DEFAULT_SEASONAL_MODEL_URL

model_name = input("Enter the model name: ")
model_dir_path = model_dir / model_name
if not model_dir_path.exists() or not model_dir_path.is_dir():
    raise ValueError(f"Model '{model_name}' not found in {model_dir}. Please choose from the available models.")
print(f"Selected model: {model_dir_path}")

croptype_head_zip_path = glob.glob(str(model_dir_path / "*.zip"))[0]
print(f"Using croptype head: {croptype_head_zip_path}")

### Step 2: Select which patch you want to run predictions on

The following cell shows you for which patches you have some prepared inputs ready.<br>

In [ ]:
inputs_dir = Path("./preprocessed_inputs")
available_dirs = glob.glob(str(inputs_dir / "*"))
for d in available_dirs:
    available_patches = glob.glob(str(Path(d) / "*" / "preprocessed-inputs_*.nc"))
    if available_patches:
        print(f"Available preprocessed input patches in {d}:")
        for patch in available_patches:
            print(f"- {Path(patch).name}")

The next cell asks you to specify the name of your patch of choice.<br>
Alternatively, you can also point to a custom .nc file...

In [ ]:
patch_nc_file = input("Enter the name of the preprocessed input patch (netCDF file): ")
patch_nc_path = Path(glob.glob(str(Path("./preprocessed_inputs/*/*") / patch_nc_file))[0])
if not patch_nc_path.exists() or not patch_nc_path.is_file() or not patch_nc_path.suffix == ".nc":
    raise ValueError(f"Preprocessed input patch '{patch_nc_file}' not found or is not a netCDF file. Please choose from the available patches.")
print(f"Selected preprocessed input patch: {patch_nc_file}")

### Step 2: Configure processing parameters

**Specify year and season**

We first show for which season your model has been trained:

In [ ]:
model_name = model_dir_path.stem
season_info = model_name.split("_")[1]
season_id = season_info.split("-")[0]
season_start = season_info.split("-")[1]
season_end = season_info.split("-")[2]
print(f"Model trained for season id: {season_id}")
print(f"Model trained for season: {season_start} - {season_end}")

Now we check what date range is available in the extracted patch and allow you to define your season of interest for inference using a slider:

In [ ]:
import pandas as pd
from notebook_utils.dateslider import date_slider

patch_name = patch_nc_path.stem
season_start_patch = pd.to_datetime(patch_name.split("_")[1])
season_end_patch = pd.to_datetime(patch_name.split("_")[2])

slider = date_slider(start_date=season_start_patch, end_date=season_end_patch, year_selector=False)

**Set other parameters**

In [ ]:
mask_cropland = True  # cropland masking using default cropland model
enable_cropland_head = True  # also export cropland rasters for QA
enable_croptype_head = True  # whether to run the croptype head at all (if False, only cropland predictions will be made)
export_class_probs = True  # export per-class probabilities in the crop type product

# Post-processing parameters
croptype_postprocess_enabled = True # Apply spatial cleaning to croptype classifications
croptype_postprocess_method = "majority_vote" # Available methods: "majority_vote", "smooth_probabilities"
croptype_postprocess_kernel = 3 # Kernel size for post-processing filter (must be odd integer, the higher the more fierce the cleaning)

cropland_postprocess_enabled = True # Apply spatial cleaning to cropland classifications
cropland_postprocess_method = "majority_vote"
cropland_postprocess_kernel = 3


**Run local inference**

The output is written as a GeoTIFF under `./local_inference/<model_name>/<patch_name>` with a filename that includes the season for which the model was run.

In [ ]:
import json
import xarray as xr
from pyproj import CRS

from notebook_utils.local_inference import (
    run_seasonal_inference,
    build_postprocess_spec,
    classification_to_geotiff,
)

# Prepare post-processing parameters
croptype_postprocess = build_postprocess_spec(
    enabled=croptype_postprocess_enabled,
    method=croptype_postprocess_method,
    kernel_size=croptype_postprocess_kernel,
)
cropland_postprocess = build_postprocess_spec(
    enabled=cropland_postprocess_enabled,
    method=cropland_postprocess_method,
    kernel_size=cropland_postprocess_kernel,
)

# Get season from slider
selected = slider.get_selection()
season_window = selected.season_window
start_season = str(season_window.start_date)
end_season = str(season_window.end_date)
season_windows = {
    season_id: (start_season, end_season)
}

# Run inference
classification_result = run_seasonal_inference(
    patch_nc_path,
    seasonal_model_zip=seasonal_model_zip_path,
    croptype_head_zip=croptype_head_zip_path,
    season_ids=[season_id],
    season_windows=season_windows,
    enforce_cropland_gate=mask_cropland,
    export_class_probabilities=export_class_probs,
    enable_cropland_head=enable_cropland_head,
    enable_croptype_head=enable_croptype_head,
    croptype_postprocess=croptype_postprocess,
    cropland_postprocess=cropland_postprocess,
    as_dataset=False,  # DataArray for GeoTIFF
)

# Specify output directory and name for the classification result
output_dir = Path("./local_inference")
classification_result_path = output_dir / model_name / patch_name / f"croptype_{start_season}_{end_season}.tif"
classification_result_path.parent.mkdir(parents=True, exist_ok=True)

# get model class names from the model config file
head_config_path = model_dir_path / "config.json"
if not head_config_path.exists():
    raise FileNotFoundError(
        f"Torch head config not found at {head_config_path}."
    )
with head_config_path.open() as fp:
    head_config = json.load(fp)
class_map = {i: name for i, name in enumerate(head_config["classes_list"])}

# retrieve epsg code from the input patch for geotiff export
with xr.open_dataset(patch_nc_path) as ds:
    epsg = CRS.from_wkt(ds.crs.attrs["spatial_ref"]).to_epsg()  

# Finally, save to geotiff
classification_to_geotiff(classification_result, epsg, classification_result_path, class_map)

**Visualize results**

You can inspect the resulting GeoTiff using QGIS, or use the cell below for quick visualization.

In [ ]:
from notebook_utils.local_inference import isolate_cropland_croptype_products
from notebook_utils.visualization import visualize_products

# split one raster into separate cropland and crop type products
products = isolate_cropland_croptype_products(
    classification_result_path,
    enable_cropland_head)

# Get class names from your model and convert to look-up table compatible with visualization function
lut_croptype = {v: k for k, v in sorted(class_map.items(), key=lambda kv: kv[0])}
luts = {'croptype': lut_croptype}

# Run simple visualization of the classification bands
visualize_products(products, luts=luts)

Congratulations, you have reached the end of this demo!

Once you are satisfied with your model, you can use the <a href="https://github.com/WorldCereal/worldcereal-classification/tree/main/notebooks/worldcereal_classification_app.ipynb" target="_blank">classification app</a> again to run your model over any area and season of interest on CDSE.